# Week 4 Exercise — Docstring & Unit Test Generator

**By:** Jaymineh  
**API:** Z.AI (GLM) + OpenRouter (frontier models)

A tool that takes raw Python code and uses LLMs to:
1. **Add Google-style docstrings** to every function and class
2. **Generate pytest unit tests** covering the code
3. **Compare quality** across multiple models side by side

**Week 4 skills demonstrated:**
- Code generation with structured prompts and multiple models
- Business metric: evaluating which model produces the most thorough docstrings and tests
- Gradio UI with tabs, streaming, model comparison, and file downloads
- Using frontier models via Z.AI and OpenRouter

In [ ]:
import os
import json
import tempfile
import time
import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv

In [ ]:
load_dotenv(override=True)

zai_api_key = os.getenv("ZAI_API_KEY")
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

if zai_api_key:
    print(f"ZAI_API_KEY found, begins: {zai_api_key[:8]}...")
else:
    print("ZAI_API_KEY not found")

if openrouter_api_key:
    print(f"OPENROUTER_API_KEY found, begins: {openrouter_api_key[:8]}...")
else:
    print("OPENROUTER_API_KEY not found")

In [ ]:
zai_client = OpenAI(api_key=zai_api_key, base_url="https://api.z.ai/api/paas/v4/")
openrouter_client = OpenAI(api_key=openrouter_api_key, base_url="https://openrouter.ai/api/v1")

In [ ]:
MODELS = {
    "GLM-4.7 (Z.AI)":            (zai_client,        "glm-4.7"),
    "GLM-5 (Z.AI)":              (zai_client,        "glm-5"),
    "Claude Sonnet 4.6":      (openrouter_client, "anthropic/claude-sonnet-4-6"),
    "GPT-5.3-Codex":          (openrouter_client, "openai/gpt-5.3-codex"),
    "MiniMax M2.5":           (openrouter_client, "minimax/minimax-m2.5"),
    "Devstral 2":           (openrouter_client, "mistralai/devstral-2512"),
}

In [ ]:
DOCSTRING_SYSTEM = """\
You are an expert Python developer. Your task is to add comprehensive Google-style \
docstrings to every function, method, and class in the provided code.

Rules:
- Preserve ALL original code exactly. Do not rename, refactor, or remove anything.
- Add a module-level docstring at the top summarising the file's purpose.
- For each function/method, document: purpose, Args, Returns, Raises (if applicable).
- For classes, document the class purpose and any __init__ parameters.
- Use Google-style docstring format (Args:, Returns:, Raises: sections).
- Add inline comments only where logic is non-obvious.
- Return ONLY the complete Python code with docstrings added. No markdown fences.
"""

UNITTEST_SYSTEM = """\
You are an expert Python test engineer. Your task is to write thorough pytest unit tests \
for the provided code.

Rules:
- Write tests using pytest (not unittest).
- Cover: normal cases, edge cases, boundary values, and error handling.
- Use descriptive test function names following test_<function>_<scenario> pattern.
- Use pytest fixtures where setup is needed.
- Use parametrize for testing multiple inputs.
- Include at least 2-3 tests per function in the source code.
- Add brief docstrings to each test explaining what it verifies.
- Return ONLY the complete test file. No markdown fences.
- Assume the source code is importable from a module called 'source'.
"""

In [ ]:
def call_model(model_name: str, system_prompt: str, code: str, stream: bool = True):
    """Call the selected model and return the response, optionally streaming."""
    client, model_id = MODELS[model_name]
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": code},
    ]

    if stream:
        response = client.chat.completions.create(
            model=model_id, messages=messages, temperature=0.2, stream=True,
        )
        result = ""
        for chunk in response:
            if not chunk.choices:
                continue
            delta = chunk.choices[0].delta
            if delta.content:
                result += delta.content
                yield result
    else:
        response = client.chat.completions.create(
            model=model_id, messages=messages, temperature=0.2,
        )
        yield response.choices[0].message.content

In [ ]:
def clean_response(text: str) -> str:
    """Strip markdown code fences if the model wraps its response."""
    cleaned = text.strip()
    if cleaned.startswith("```"):
        lines = cleaned.split("\n")
        cleaned = "\n".join(lines[1:])
    if cleaned.endswith("```"):
        cleaned = cleaned[:-3].rstrip()
    return cleaned

In [ ]:
# Quick test: generate docstrings for a small function with GLM-4.7

test_code = '''
def fibonacci(n):
    if n <= 0:
        return []
    elif n == 1:
        return [0]
    seq = [0, 1]
    for i in range(2, n):
        seq.append(seq[-1] + seq[-2])
    return seq
'''

for chunk in call_model("GLM-4.7 (Z.AI)", DOCSTRING_SYSTEM, test_code, stream=False):
    result = chunk
print(clean_response(result))

In [ ]:
# Quick test: generate unit tests for the same function

for chunk in call_model("GLM-4.7 (Z.AI)", UNITTEST_SYSTEM, test_code, stream=False):
    result = chunk
print(clean_response(result))

In [ ]:
def generate_docstrings(code, model_name):
    """Gradio callback for the Docstrings tab (streaming)."""
    for chunk in call_model(model_name, DOCSTRING_SYSTEM, code, stream=True):
        yield clean_response(chunk)


def generate_tests(code, model_name):
    """Gradio callback for the Unit Tests tab (streaming)."""
    for chunk in call_model(model_name, UNITTEST_SYSTEM, code, stream=True):
        yield clean_response(chunk)


def save_to_file(content, filename):
    """Write content to a temp file and return the path for Gradio download."""
    path = os.path.join(tempfile.gettempdir(), filename)
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)
    return path


def download_docstrings(content):
    return save_to_file(content, "documented_code.py")


def download_tests(content):
    return save_to_file(content, "test_source.py")

In [ ]:
def compare_models(code, task):
    """Run two models (GLM-5 and Claude Sonnet) on the same task and return both results + timing."""
    system_prompt = DOCSTRING_SYSTEM if task == "Docstrings" else UNITTEST_SYSTEM
    model_a = "GLM-5 (Z.AI)"
    model_b = "Claude Sonnet 4.6"

    results = {}
    for name in [model_a, model_b]:
        start = time.time()
        output = ""
        for chunk in call_model(name, system_prompt, code, stream=False):
            output = chunk
        elapsed = time.time() - start
        results[name] = (clean_response(output), elapsed)

    a_text, a_time = results[model_a]
    b_text, b_time = results[model_b]

    summary = (
        f"**{model_a}:** {len(a_text.splitlines())} lines, {a_time:.1f}s\n\n"
        f"**{model_b}:** {len(b_text.splitlines())} lines, {b_time:.1f}s"
    )

    return a_text, b_text, summary

In [ ]:
SAMPLE_CODE = '''def calculate_discount(price, discount_percent, is_member=False):
    if price < 0 or discount_percent < 0 or discount_percent > 100:
        raise ValueError("Invalid price or discount")
    discount = price * (discount_percent / 100)
    if is_member:
        discount += price * 0.05
    final = price - discount
    return max(final, 0)


def bulk_discount(items):
    total = 0
    for name, price, qty in items:
        subtotal = price * qty
        if qty >= 10:
            subtotal *= 0.9
        elif qty >= 5:
            subtotal *= 0.95
        total += subtotal
    return round(total, 2)


def parse_order(order_string):
    orders = []
    for line in order_string.strip().split("\\n"):
        parts = line.split(",")
        if len(parts) != 3:
            continue
        name = parts[0].strip()
        price = float(parts[1].strip())
        qty = int(parts[2].strip())
        orders.append((name, price, qty))
    return orders
'''

In [ ]:
model_names = list(MODELS.keys())

with gr.Blocks(theme=gr.themes.Soft(), title="Docstring & Unit Test Generator") as demo:
    gr.Markdown("# Docstring & Unit Test Generator")
    gr.Markdown(
        "Paste Python code below. Choose a model and generate **Google-style docstrings** "
        "or **pytest unit tests**. Use the Compare tab to evaluate two models side by side."
    )

    with gr.Tabs():

        # ---- Tab 1: Docstrings ----
        with gr.TabItem("Docstrings"):
            with gr.Row():
                with gr.Column():
                    doc_input = gr.Code(
                        label="Input Python Code", language="python",
                        value=SAMPLE_CODE, lines=18,
                    )
                    doc_model = gr.Dropdown(choices=model_names, value=model_names[0], label="Model")
                    doc_btn = gr.Button("Generate Docstrings", variant="primary")
                with gr.Column():
                    doc_output = gr.Code(label="Documented Code", language="python", lines=25)
                    doc_dl_btn = gr.Button("Download .py")
                    doc_file = gr.File(label="Download", visible=False)

            doc_btn.click(fn=generate_docstrings, inputs=[doc_input, doc_model], outputs=doc_output)
            doc_dl_btn.click(
                fn=download_docstrings, inputs=doc_output,
                outputs=doc_file,
            ).then(lambda: gr.update(visible=True), outputs=doc_file)

        # ---- Tab 2: Unit Tests ----
        with gr.TabItem("Unit Tests"):
            with gr.Row():
                with gr.Column():
                    test_input = gr.Code(
                        label="Input Python Code", language="python",
                        value=SAMPLE_CODE, lines=18,
                    )
                    test_model = gr.Dropdown(choices=model_names, value=model_names[0], label="Model")
                    test_btn = gr.Button("Generate Tests", variant="primary")
                with gr.Column():
                    test_output = gr.Code(label="Generated Tests", language="python", lines=25)
                    test_dl_btn = gr.Button("Download test_.py")
                    test_file = gr.File(label="Download", visible=False)

            test_btn.click(fn=generate_tests, inputs=[test_input, test_model], outputs=test_output)
            test_dl_btn.click(
                fn=download_tests, inputs=test_output,
                outputs=test_file,
            ).then(lambda: gr.update(visible=True), outputs=test_file)

        # ---- Tab 3: Compare Models ----
        with gr.TabItem("Compare Models"):
            gr.Markdown(
                "Send the same code to **GLM-5 (Z.AI)** and **Claude Sonnet 4.6 (OpenRouter)** "
                "and compare their output quality, verbosity, and speed."
            )
            cmp_input = gr.Code(
                label="Input Python Code", language="python",
                value=SAMPLE_CODE, lines=15,
            )
            cmp_task = gr.Radio(
                choices=["Docstrings", "Unit Tests"], value="Docstrings", label="Task",
            )
            cmp_btn = gr.Button("Compare", variant="primary")

            cmp_summary = gr.Markdown(label="Summary")
            with gr.Row():
                with gr.Column():
                    gr.Markdown("### GLM-5 (Z.AI)")
                    cmp_a = gr.Code(language="python", lines=25)
                with gr.Column():
                    gr.Markdown("### Claude Sonnet 4.6 (OpenRouter)")
                    cmp_b = gr.Code(language="python", lines=25)

            cmp_btn.click(
                fn=compare_models, inputs=[cmp_input, cmp_task],
                outputs=[cmp_a, cmp_b, cmp_summary],
            )

demo.launch()